# 法律问答 Qwen2.5-1.5B LoRA 微调全流程（Colab T4）

**使用方法**：把本项目的 zip 包上传到 Colab（左侧文件面板 → 上传 `legal-qa-sft.zip`），然后从上到下依次运行。

| 阶段 | 预计耗时（T4） |
|---|---|
| 环境安装 | 3-5 分钟 |
| 数据构建 | 2-3 分钟 |
| 冒烟（0.5B 训练+推理） | ~15 分钟 |
| 正式训练（1.5B × 3 epochs） | 30-40 分钟 |
| 批量推理（微调 + 基线） | 25-35 分钟 |
| 评测 | 2-5 分钟（BERTScore 首次多 3-5 分钟） |
| 合并 + GGUF 导出 | 5-8 分钟 |

合计约 1.5-2 小时，建议在 Colab 会话时限内一次跑完，跑完立刻保存到 Drive。

In [ ]:
!nvidia-smi
!unzip -o -q legal-qa-sft.zip -d /content/
%cd /content/legal-qa-sft
!python -m py_compile data/make_dataset.py eval/metrics.py eval/robustness.py deploy/benchmark_latency.py deploy/gradio_app.py && echo '===== 语法自测通过 ====='

In [ ]:
# 安装依赖（llamafactory 安装约 3-5 分钟，可能出现依赖告警，可忽略）
%pip install -q llamafactory rouge-chinese sacrebleu jieba bert-score
import llamafactory
print('llamafactory', llamafactory.__version__)

In [ ]:
# 阶段 1：数据构建（下载 DISC-Law-SFT 约 300MB → 清洗 → 5000/300/300 切分）
!python data/make_dataset.py --train 5000 --val 300 --test 300

In [ ]:
# 阶段 2：冒烟训练（0.5B + 500 样本，约 10 分钟）
!llamafactory-cli train configs/smoke_0.5b.yaml

In [ ]:
# 阶段 2：冒烟推理 + 评测（验证 训练→推理→评测 全链路）
!llamafactory-cli train configs/infer_smoke.yaml
!python eval/metrics.py --pred results/smoke-0.5b/generated_predictions.jsonl

In [ ]:
# 阶段 3：正式训练 Qwen2.5-1.5B（约 30-40 分钟）
# 关注每个 epoch 的 eval_loss：持续上升说明过拟合，可把 num_train_epochs 降为 2.0
!llamafactory-cli train configs/train_1.5b.yaml

In [ ]:
# 阶段 4：批量推理 —— 微调后模型 + 基座 zero-shot 基线（共约 25-35 分钟）
!llamafactory-cli train configs/infer_1.5b.yaml
!llamafactory-cli train configs/infer_baseline_1.5b.yaml

In [ ]:
# 阶段 4：评测对比（BERTScore 首次会下载中文 BERT 约 400MB）
!python eval/metrics.py --pred results/ft-1.5b/generated_predictions.jsonl --bertscore --out eval_results/ft-1.5b.json
!python eval/metrics.py --pred results/base-1.5b/generated_predictions.jsonl --bertscore --out eval_results/base-1.5b.json

import json
ft = json.load(open('eval_results/ft-1.5b.json', encoding='utf-8'))
base = json.load(open('eval_results/base-1.5b.json', encoding='utf-8'))
print(f"{'指标':<18}{'基座 zero-shot':>14}{'微调后':>12}")
print(f"{'ROUGE-L':<20}{base['rouge_l']:>12.4f}{ft['rouge_l']:>12.4f}")
print(f"{'BLEU (char)':<20}{base['bleu_char']:>12.2f}{ft['bleu_char']:>12.2f}")
print(f"{'BERTScore F1':<20}{base.get('bertscore_f1', 0):>12.4f}{ft.get('bertscore_f1', 0):>12.4f}")
print(f"{'引用格式合规率':<19}{base['citation']['format_rate']:>12.2%}{ft['citation']['format_rate']:>12.2%}")
print(f"{'引用事实正确率':<19}{base['citation']['precision']:>12.2%}{ft['citation']['precision']:>12.2%}")
print(f"{'引用完全一致率':<19}{base['citation']['exact_match']:>12.2%}{ft['citation']['exact_match']:>12.2%}")

In [ ]:
# 阶段 4.5：鲁棒性测试（60 条 = 20 × 3 扰动，约 5 分钟）
!python eval/robustness.py --n 20
!llamafactory-cli train configs/infer_robustness.yaml
!python eval/metrics.py --pred results/ft-1.5b-robust/generated_predictions.jsonl --out eval_results/ft-1.5b-robust.json
print('对比干净测试集的引用格式合规率，观察三类扰动下的退化幅度')

In [ ]:
# 阶段 5：合并 LoRA → 导出 GGUF（Q8_0 约 1.6GB）
!llamafactory-cli export configs/merge_1.5b.yaml
!bash deploy/convert_gguf.sh

In [ ]:
# 阶段 5（可选）：Colab 上直接压测 GGUF（CUDA 版 llama-cpp-python 编译约 5-10 分钟）
# 也可以跳过此格，把 GGUF 下载回本地 Windows 后再压测
!CMAKE_ARGS='-DGGML_CUDA=on' pip install -q llama-cpp-python
!python deploy/benchmark_latency.py --model models/qwen2.5-1.5b-legal-q8_0.gguf -n 30 --n-gpu-layers -1

In [ ]:
# 收尾：全部产物保存到 Google Drive（Colab 会话会断，务必执行）
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/legal-qa-sft-output
!cp -r saves /content/drive/MyDrive/legal-qa-sft-output/ 2>/dev/null || true
!cp -r results /content/drive/MyDrive/legal-qa-sft-output/ 2>/dev/null || true
!cp -r eval_results /content/drive/MyDrive/legal-qa-sft-output/ 2>/dev/null || true
!cp models/*.gguf /content/drive/MyDrive/legal-qa-sft-output/ 2>/dev/null || true
!cp data/data_stats.json /content/drive/MyDrive/legal-qa-sft-output/ 2>/dev/null || true
print('已保存到 Drive: legal-qa-sft-output/')